# 🛡️ TrustCart — Fake Review Detection (ML Module)

This notebook trains a machine learning model to classify Amazon product reviews as:
- **CG (Computer Generated)** → Fake review
- **OR (Original Review)** → Genuine review

This ML module powers the fake review detection layer inside the TrustCart AI pipeline.

**Dataset:** 40,432 Amazon reviews (perfectly balanced — 20,216 fake / 20,216 genuine)

**Pipeline:**
1. EDA & Visualization
2. Feature Engineering (text + statistical features)
3. Model Training (Logistic Regression + XGBoost)
4. Evaluation (Accuracy, F1, Confusion Matrix, ROC)
5. Save model for production use

## 1. Import Libraries

In [ ]:
# ── Core
import pandas as pd
import numpy as np
import re
import joblib
import warnings
warnings.filterwarnings('ignore')

# ── Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from wordcloud import WordCloud

# ── NLP
from sklearn.feature_extraction.text import TfidfVectorizer

# ── ML
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix
from xgboost import XGBClassifier

print("✅ All libraries loaded successfully")
print(f"Pandas: {pd.__version__} | NumPy: {np.__version__}")

## 2. Load & Explore Data (EDA)

In [ ]:
# ── Load dataset
# Update path if running locally
df = pd.read_csv('fake_reviews_dataset.csv')

print("📦 Dataset Shape:", df.shape)
print("\n📋 Columns:", df.columns.tolist())
print("\n🔍 Data Types:")
print(df.dtypes)
print("\n❓ Missing Values:")
print(df.isnull().sum())
df.head(5)

In [ ]:
# ── Label distribution
print("🏷️  Label Distribution:")
print(df['label'].value_counts())
print("\nCG = Computer Generated (FAKE) | OR = Original Review (GENUINE)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
colors = ['#e74c3c', '#2ecc71']
df['label'].value_counts().plot(
    kind='bar', ax=axes[0], color=colors, edgecolor='black', width=0.5
)
axes[0].set_title('Review Label Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Label (CG=Fake, OR=Genuine)')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{int(bar.get_height()):,}', ha='center', fontsize=11)

# Pie chart
df['label'].value_counts().plot(
    kind='pie', ax=axes[1], colors=colors,
    autopct='%1.1f%%', startangle=90,
    labels=['Fake (CG)', 'Genuine (OR)'],
    textprops={'fontsize': 12}
)
axes[1].set_title('Fake vs Genuine Ratio', fontsize=13, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('eda_label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Perfectly balanced dataset — no class imbalance issues!")

In [ ]:
# ── Rating distribution by label
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (label, group), title, color in zip(
    axes,
    df.groupby('label')['rating'],
    ['Fake Reviews (CG)', 'Genuine Reviews (OR)'],
    ['#e74c3c', '#2ecc71']
):
    counts = group.value_counts().sort_index()
    ax.bar(counts.index, counts.values, color=color, edgecolor='black', alpha=0.85)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Star Rating')
    ax.set_ylabel('Count')
    ax.set_xticks([1, 2, 3, 4, 5])

plt.suptitle('Rating Distribution: Fake vs Genuine Reviews', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_rating_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Review length analysis
df['review_length'] = df['text_'].apply(lambda x: len(str(x)))
df['word_count'] = df['text_'].apply(lambda x: len(str(x).split()))

print("📏 Review Length Stats by Label:")
print(df.groupby('label')[['review_length', 'word_count']].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for col, ax, title in zip(
    ['review_length', 'word_count'],
    axes,
    ['Review Character Length', 'Word Count per Review']
):
    for label, color in [('CG', '#e74c3c'), ('OR', '#2ecc71')]:
        subset = df[df['label'] == label][col]
        subset.clip(upper=subset.quantile(0.97)).plot(
            kind='hist', bins=40, alpha=0.6, color=color,
            label=f'{"Fake" if label=="CG" else "Genuine"} ({label})', ax=ax
        )
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel(col.replace('_', ' ').title())
    ax.set_ylabel('Frequency')
    ax.legend()

plt.suptitle('Review Length: Fake vs Genuine', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_review_length.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Word Clouds: Fake vs Genuine
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, label, title, cmap in zip(
    axes,
    ['CG', 'OR'],
    ['🔴 Fake Reviews Word Cloud', '🟢 Genuine Reviews Word Cloud'],
    ['Reds', 'Greens']
):
    text = ' '.join(df[df['label'] == label]['text_'].astype(str).tolist())
    wc = WordCloud(
        width=700, height=400,
        background_color='white',
        colormap=cmap,
        max_words=100
    ).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')

plt.suptitle('Most Common Words in Fake vs Genuine Reviews', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Category breakdown
plt.figure(figsize=(14, 5))
cat_label = df.groupby(['category', 'label']).size().unstack(fill_value=0)
cat_label.plot(kind='bar', color=['#e74c3c', '#2ecc71'], edgecolor='black',
               figsize=(14, 5), alpha=0.85)
plt.title('Fake vs Genuine Reviews by Product Category', fontsize=13, fontweight='bold')
plt.xlabel('Category')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.legend(['Fake (CG)', 'Genuine (OR)'])
plt.tight_layout()
plt.savefig('eda_category_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Engineering

We extract **two types of features** that directly mirror TrustCart's 8 heuristics:

| Feature | Maps to TrustCart Heuristic |
|---|---|
| TF-IDF of review text | Lexical pattern detection |
| Review length | Unusually short/long reviews |
| Word count | Padding / verbosity detection |
| Exclamation count | Over-enthusiasm signal |
| Capital word ratio | Fake emphasis patterns |
| Avg word length | Vocabulary sophistication |
| Unique word ratio | Repetition / copy-paste detection |
| Rating (as feature) | Rating-text mismatch signal |

In [ ]:
def extract_statistical_features(text_series):
    """
    Extract 8 statistical heuristic features from review text.
    These directly mirror TrustCart's fake review detection logic.
    """
    features = pd.DataFrame()

    # Heuristic 1: Review character length
    features['char_length'] = text_series.apply(lambda x: len(str(x)))

    # Heuristic 2: Word count
    features['word_count'] = text_series.apply(lambda x: len(str(x).split()))

    # Heuristic 3: Exclamation mark count (over-enthusiasm signal)
    features['exclamation_count'] = text_series.apply(lambda x: str(x).count('!'))

    # Heuristic 4: Ratio of UPPERCASE words (fake emphasis)
    features['caps_ratio'] = text_series.apply(
        lambda x: sum(1 for w in str(x).split() if w.isupper()) / max(len(str(x).split()), 1)
    )

    # Heuristic 5: Average word length (vocabulary complexity)
    features['avg_word_length'] = text_series.apply(
        lambda x: np.mean([len(w) for w in str(x).split()]) if str(x).split() else 0
    )

    # Heuristic 6: Unique word ratio (copy-paste / repetition detection)
    features['unique_word_ratio'] = text_series.apply(
        lambda x: len(set(str(x).lower().split())) / max(len(str(x).split()), 1)
    )

    # Heuristic 7: Sentence count
    features['sentence_count'] = text_series.apply(
        lambda x: len(re.split(r'[.!?]+', str(x)))
    )

    # Heuristic 8: Question mark count (genuine reviews ask questions more)
    features['question_count'] = text_series.apply(lambda x: str(x).count('?'))

    return features


# ── Encode target label: CG (fake) = 1, OR (genuine) = 0
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])  # CG=0, OR=1
# Make CG (fake) = 1 for intuitive interpretation
df['label_encoded'] = 1 - df['label_encoded']  # CG=1 (fake), OR=0 (genuine)

print("🏷️  Encoded labels:")
print(df[['label', 'label_encoded']].value_counts())
print("\n1 = Fake (CG) | 0 = Genuine (OR)")

In [ ]:
# ── Extract statistical features
stat_features = extract_statistical_features(df['text_'])

# ── Add rating as a feature
stat_features['rating'] = df['rating'].values

print("📊 Statistical Features Shape:", stat_features.shape)
print("\nFeature Preview:")
stat_features.head()

In [ ]:
# ── Feature correlation heatmap
corr_df = stat_features.copy()
corr_df['is_fake'] = df['label_encoded'].values

plt.figure(figsize=(10, 7))
mask = np.triu(np.ones_like(corr_df.corr(), dtype=bool))
sns.heatmap(
    corr_df.corr(), annot=True, fmt='.2f',
    cmap='RdYlGn', center=0,
    mask=mask, linewidths=0.5,
    annot_kws={'size': 9}
)
plt.title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── TF-IDF vectorization of review text
# We use top 5000 features + unigrams & bigrams
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),       # unigrams + bigrams
    stop_words='english',
    min_df=3,                 # ignore very rare terms
    sublinear_tf=True         # log-scale TF
)

# ── Train/test split (before fitting TF-IDF to avoid leakage)
X_stat = stat_features.values
y = df['label_encoded'].values

(
    X_stat_train, X_stat_test,
    X_text_train_raw, X_text_test_raw,
    y_train, y_test
) = train_test_split(
    X_stat, df['text_'].values, y,
    test_size=0.2, random_state=42, stratify=y
)

# Fit TF-IDF ONLY on training data
X_tfidf_train = tfidf.fit_transform(X_text_train_raw)
X_tfidf_test = tfidf.transform(X_text_test_raw)

# ── Combine TF-IDF + statistical features
X_train = hstack([X_tfidf_train, csr_matrix(X_stat_train)])
X_test = hstack([X_tfidf_test, csr_matrix(X_stat_test)])

print(f"✅ Training set: {X_train.shape}")
print(f"✅ Test set:     {X_test.shape}")
print(f"\nTrain label distribution: Fake={y_train.sum()}, Genuine={len(y_train)-y_train.sum()}")
print(f"Test label distribution:  Fake={y_test.sum()}, Genuine={len(y_test)-y_test.sum()}")

## 4. Model Training

In [ ]:
# ──────────────────────────────────────────────
# Model 1: Logistic Regression (fast baseline)
# ──────────────────────────────────────────────
print("🔵 Training Logistic Regression...")

lr_model = LogisticRegression(
    max_iter=1000,
    C=1.0,
    solver='saga',
    random_state=42,
    n_jobs=-1
)
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)
lr_proba = lr_model.predict_proba(X_test)[:, 1]

lr_acc = accuracy_score(y_test, lr_preds)
lr_f1 = f1_score(y_test, lr_preds)
lr_auc = roc_auc_score(y_test, lr_proba)

print(f"\n✅ Logistic Regression Results:")
print(f"   Accuracy : {lr_acc:.4f} ({lr_acc*100:.2f}%)")
print(f"   F1 Score : {lr_f1:.4f}")
print(f"   ROC-AUC  : {lr_auc:.4f}")

In [ ]:
# ──────────────────────────────────────────────
# Model 2: XGBoost (production model)
# ──────────────────────────────────────────────
print("🟠 Training XGBoost...")

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    tree_method='hist'  # faster for large datasets
)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

xgb_acc = accuracy_score(y_test, xgb_preds)
xgb_f1 = f1_score(y_test, xgb_preds)
xgb_auc = roc_auc_score(y_test, xgb_proba)

print(f"\n✅ XGBoost Results:")
print(f"   Accuracy : {xgb_acc:.4f} ({xgb_acc*100:.2f}%)")
print(f"   F1 Score : {xgb_f1:.4f}")
print(f"   ROC-AUC  : {xgb_auc:.4f}")

## 5. Model Evaluation

In [ ]:
# ── Model comparison table
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'XGBoost'],
    'Accuracy': [lr_acc, xgb_acc],
    'F1 Score': [lr_f1, xgb_f1],
    'ROC-AUC':  [lr_auc, xgb_auc]
})
print("📊 Model Comparison:")
print(results.to_string(index=False))

best_model = 'XGBoost' if xgb_f1 >= lr_f1 else 'Logistic Regression'
print(f"\n🏆 Best Model: {best_model}")

In [ ]:
# ── Confusion matrices + ROC curves (side by side)
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

models_info = [
    ('Logistic Regression', lr_preds, lr_proba, '#3498db'),
    ('XGBoost',             xgb_preds, xgb_proba, '#e67e22'),
]

for col, (name, preds, proba, color) in enumerate(models_info):
    # Confusion Matrix
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        ax=axes[0][col],
        xticklabels=['Genuine (0)', 'Fake (1)'],
        yticklabels=['Genuine (0)', 'Fake (1)'],
        linewidths=1
    )
    acc = accuracy_score(y_test, preds)
    f1  = f1_score(y_test, preds)
    axes[0][col].set_title(
        f'{name} — Confusion Matrix\nAcc: {acc:.3f} | F1: {f1:.3f}',
        fontsize=11, fontweight='bold'
    )
    axes[0][col].set_ylabel('Actual')
    axes[0][col].set_xlabel('Predicted')

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    axes[1][col].plot(fpr, tpr, color=color, lw=2, label=f'ROC (AUC = {auc:.3f})')
    axes[1][col].plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
    axes[1][col].fill_between(fpr, tpr, alpha=0.1, color=color)
    axes[1][col].set_title(f'{name} — ROC Curve', fontsize=11, fontweight='bold')
    axes[1][col].set_xlabel('False Positive Rate')
    axes[1][col].set_ylabel('True Positive Rate')
    axes[1][col].legend()
    axes[1][col].grid(alpha=0.3)

plt.suptitle('Model Evaluation: Logistic Regression vs XGBoost', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Detailed classification report
print("=" * 50)
print("📋 LOGISTIC REGRESSION — Classification Report")
print("=" * 50)
print(classification_report(y_test, lr_preds, target_names=['Genuine (OR)', 'Fake (CG)']))

print("=" * 50)
print("📋 XGBOOST — Classification Report")
print("=" * 50)
print(classification_report(y_test, xgb_preds, target_names=['Genuine (OR)', 'Fake (CG)']))

In [ ]:
# ── Top TF-IDF features for Logistic Regression
feature_names = tfidf.get_feature_names_out().tolist() + list(stat_features.columns)
coefs = lr_model.coef_[0]

top_fake    = np.argsort(coefs)[-15:][::-1]
top_genuine = np.argsort(coefs)[:15]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(
    [feature_names[i] for i in top_fake],
    coefs[top_fake], color='#e74c3c', edgecolor='black'
)
axes[0].set_title('Top 15 Features → Fake Reviews', fontsize=11, fontweight='bold')
axes[0].invert_yaxis()

axes[1].barh(
    [feature_names[i] for i in top_genuine],
    np.abs(coefs[top_genuine]), color='#2ecc71', edgecolor='black'
)
axes[1].set_title('Top 15 Features → Genuine Reviews', fontsize=11, fontweight='bold')
axes[1].invert_yaxis()

plt.suptitle('Most Influential Features (Logistic Regression Coefficients)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Save Models for Production

In [ ]:
# ── Save both models and TF-IDF vectorizer
joblib.dump(xgb_model,  'xgboost_fake_review_model.pkl')
joblib.dump(lr_model,   'logistic_regression_model.pkl')
joblib.dump(tfidf,      'tfidf_vectorizer.pkl')

print("✅ Models saved:")
print("   xgboost_fake_review_model.pkl")
print("   logistic_regression_model.pkl")
print("   tfidf_vectorizer.pkl")

In [ ]:
# ── Quick inference test
def predict_review(review_text, rating=5):
    """
    Predict if a review is fake or genuine.
    Returns: label, confidence score (0-1)
    """
    # TF-IDF
    tfidf_feat = tfidf.transform([review_text])

    # Statistical features
    stat_feat = extract_statistical_features(pd.Series([review_text]))
    stat_feat['rating'] = rating

    # Combine
    X = hstack([tfidf_feat, csr_matrix(stat_feat.values)])

    prob = xgb_model.predict_proba(X)[0][1]
    label = 'FAKE 🔴' if prob >= 0.5 else 'GENUINE 🟢'
    return label, round(prob, 4)


# Test reviews
test_reviews = [
    ("Great product! Very nice. Good quality. I love it! Amazing purchase!!", 5),
    ("I've been using this for 3 months. The build quality is solid but the battery life decreased after 2 months of heavy use. Still, decent value for the price.", 4),
    ("Best product ever!! 5 stars!! Everyone should buy this!! Perfect!!", 5),
    ("The product arrived with a small dent on the corner. Customer support was responsive and sent a replacement within 2 days. The item itself works as advertised.", 3),
]

print("🔍 Inference Test:\n")
print(f"{'Review':<70} {'Rating':>6} {'Prediction':<15} {'Fake Prob':>9}")
print("-" * 110)
for review, rating in test_reviews:
    label, prob = predict_review(review, rating)
    print(f"{review[:67]:<70} {rating:>6} {label:<15} {prob:>9.4f}")

## 7. Summary

| Metric | Logistic Regression | XGBoost |
|--------|--------------------|---------|
| Accuracy | See above | See above |
| F1 Score | See above | See above |
| ROC-AUC | See above | See above |

### Key Findings from EDA:
- Fake reviews tend to use more exclamation marks and ALL CAPS words
- Fake reviews cluster heavily at 5-star ratings
- Genuine reviews have higher vocabulary diversity (unique word ratio)
- Fake reviews are often shorter with repetitive phrasing

### Integration with TrustCart:
The saved `xgboost_fake_review_model.pkl` and `tfidf_vectorizer.pkl` can be loaded in a Python microservice that TrustCart's Node.js backend calls via REST API — replacing or augmenting the existing heuristic-only detection.

### Next Steps / Improvements:
- Fine-tune a BERT model for higher accuracy
- Add cross-validation with k-fold
- Build a FastAPI wrapper around the saved model
- Collect domain-specific Indian e-commerce review data